<a href="https://colab.research.google.com/github/JE7500/Semicon.AI/blob/main/semicon_mvp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install required packages
!pip install transformers torch gradio accelerate peft bitsandbytes

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Path to your restored adapter folder
adapter_path = "/content/drive/MyDrive/codellama-7b-verilog-lora"

# Load base model in 4‑bit
base_model_name = "codellama/CodeLlama-7b-hf"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(base_model_name)
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16
)

# Load your restored adapter
model = PeftModel.from_pretrained(base_model, adapter_path)

# Quick test to confirm it works
prompt = "4-bit counter with asynchronous reset"
inputs = tokenizer(f"### Instruction:\n{prompt}\n\n### Verilog:\n", return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=100, temperature=0.2)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

In [ ]:
import gradio as gr
import time
import torch
import numpy as np

# Assume model and tokenizer are already loaded (e.g., from a fine-tuned checkpoint)
# model = ...
# tokenizer = ...

CONFIDENCE_THRESHOLD = 0.7  # Adjust as needed

def generate_verilog_with_confidence(prompt):
    input_text = f"### Instruction:\n{prompt}\n\n### Verilog:\n"
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

    # Generate with scores to compute confidence
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.2,
            output_scores=True,
            return_dict_in_generate=True
        )

    # Decode the full generated sequence
    generated_sequence = outputs.sequences[0]
    generated_text = tokenizer.decode(generated_sequence, skip_special_tokens=True)

    # Extract only the Verilog part (after "### Verilog:")
    if "### Verilog:" in generated_text:
        code = generated_text.split("### Verilog:")[-1].strip()
    else:
        code = generated_text.strip()

    # Compute confidence: average probability of generated tokens (excluding input)
    # scores are logits for each generated step, shape (num_generated_tokens, vocab_size)
    scores = outputs.scores  # list of tensors, one per generated token
    if scores:
        # Convert logits to probabilities via softmax
        probs = [torch.softmax(score, dim=-1) for score in scores]
        # Get probability of the chosen token at each step
        chosen_token_probs = []
        for i, prob in enumerate(probs):
            # The token id at this step is in generated_sequence at position len(input_ids)+i
            token_id = generated_sequence[inputs['input_ids'].shape[1] + i]
            chosen_prob = prob[0, token_id].item()  # batch size 1
            chosen_token_probs.append(chosen_prob)
        confidence = np.mean(chosen_token_probs) if chosen_token_probs else 0.0
    else:
        confidence = 0.0  # no tokens generated (should not happen)

    return code, confidence

def infer(prompt):
    start = time.time()
    code, confidence = generate_verilog_with_confidence(prompt)
    elapsed = time.time() - start

    # Refusal if confidence is too low
    if confidence < CONFIDENCE_THRESHOLD:
        refusal_msg = (
            "⚠️ I cannot generate safe code for this specification with sufficient confidence.\n"
            f"(Confidence: {confidence:.2f} < {CONFIDENCE_THRESHOLD})\n\n"
            "Please provide more details or rephrase your request."
        )
        return f"{refusal_msg}\n\n---\n⏱️ Attempted in {elapsed:.2f} seconds"

    # Otherwise return code with confidence info
    result = f"{code}\n\n---\n✅ Confidence: {confidence:.2f}  |  ⏱️ Generated in {elapsed:.2f} seconds"
    return result

iface = gr.Interface(
    fn=infer,
    inputs=gr.Textbox(lines=3, placeholder="Describe your hardware module..."),
    outputs=gr.Code(language=None),  # Removed language="verilog"
    title="Semicon.ai - Fine‑tuned CodeLlama‑7B + LoRA",
    description="Model fine‑tuned on 18,500 Verilog modules. Confidence threshold for safe generation: 0.7",
    examples=[
        ["4-bit counter with asynchronous reset"],
        ["8-bit adder with carry in and out"],
        ["Finite state machine for a traffic light (3 states)"]
    ]
)

iface.launch(share=True)

In [ ]:
import gradio as gr
import time

def generate_verilog(prompt):
    input_text = f"### Instruction:\n{prompt}\n\n### Verilog:\n"
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=256, temperature=0.2)
    generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
    if "### Verilog:" in generated:
        return generated.split("### Verilog:")[-1].strip()
    return generated

def infer(prompt):
    start = time.time()
    code = generate_verilog(prompt)
    elapsed = time.time() - start
    return f"{code}\n\n---\n⏱️ Generated in {elapsed:.2f} seconds"

iface = gr.Interface(
    fn=infer,
    inputs=gr.Textbox(lines=3, placeholder="Describe your hardware module..."),
    outputs=gr.Code(language=None),
    title="Semicon.ai - Fine‑tuned CodeLlama‑7B + LoRA",
    description="Model fine‑tuned on 18,500 Verilog modules.",
    examples=[
        ["4-bit counter with asynchronous reset"],
        ["8-bit adder with carry in and out"],
        ["Finite state machine for a traffic light (3 states)"]
    ]
)
iface.launch(share=True)

In [ ]:
# Install Streamlit
!pip install streamlit

import streamlit as st
import time

# ------------------------------
# Page configuration
# ------------------------------
st.set_page_config(
    page_title="Semicon.ai MVP",
    page_icon="🧠",
    layout="wide",
    initial_sidebar_state="expanded"
)

# ------------------------------
# Custom CSS for a polished look
# ------------------------------
st.markdown("""
<style>
    .main-header {
        font-size: 3rem;
        font-weight: 700;
        color: #1E3A8A;
        text-align: center;
        margin-bottom: 0.2rem;
    }
    .sub-header {
        font-size: 1.2rem;
        color: #4B5563;
        text-align: center;
        margin-top: 0rem;
        margin-bottom: 2rem;
    }
    .metric-card {
        background-color: #F3F4F6;
        border-radius: 10px;
        padding: 1.5rem;
        text-align: center;
        box-shadow: 0 2px 4px rgba(0,0,0,0.1);
    }
    .metric-value {
        font-size: 2.5rem;
        font-weight: 700;
        color: #1E3A8A;
    }
    .metric-label {
        font-size: 1rem;
        color: #6B7280;
        margin-top: 0.3rem;
    }
    .footer {
        text-align: center;
        margin-top: 3rem;
        padding-top: 1rem;
        border-top: 1px solid #E5E7EB;
        color: #9CA3AF;
        font-size: 0.9rem;
    }
    .stButton>button {
        background-color: #1E3A8A;
        color: white;
        font-weight: 600;
        border-radius: 8px;
        padding: 0.5rem 2rem;
        border: none;
        transition: all 0.2s;
    }
    .stButton>button:hover {
        background-color: #2563EB;
        color: white;
        border: none;
    }
    .example-button {
        background-color: #EFF6FF;
        border: 1px solid #BFDBFE;
        border-radius: 20px;
        padding: 0.3rem 1rem;
        margin: 0.2rem;
        font-size: 0.8rem;
        cursor: pointer;
        color: #1E3A8A;
        display: inline-block;
    }
    .example-button:hover {
        background-color: #DBEAFE;
    }
</style>
""", unsafe_allow_html=True)

# ------------------------------
# Header
# ------------------------------
st.markdown('<div class="main-header">Semicon.ai</div>', unsafe_allow_html=True)
st.markdown('<div class="sub-header">Your AI Copilot for Hardware Design</div>', unsafe_allow_html=True)

# ------------------------------
# Sidebar with metrics (from PDF)
# ------------------------------
with st.sidebar:
    st.markdown("## 📊 Model Performance")
    col1, col2 = st.columns(2)
    with col1:
        st.markdown('<div class="metric-card">'
                    '<div class="metric-value">94.2%</div>'
                    '<div class="metric-label">Syntax Pass Rate</div>'
                    '</div>', unsafe_allow_html=True)
    with col2:
        st.markdown('<div class="metric-card">'
                    '<div class="metric-value">110x</div>'
                    '<div class="metric-label">Speedup vs. Human</div>'
                    '</div>', unsafe_allow_html=True)

    st.markdown("---")
    st.markdown("### 🧪 Example Prompts")
    if st.button("🔁 4-bit counter"):
        st.session_state.prompt = "Write a Verilog module for a 4-bit synchronous counter with active-low reset."
    if st.button("➕ Simple ALU"):
        st.session_state.prompt = "Design an 8-bit ALU that supports add, subtract, AND, OR."
    if st.button("📟 UART transmitter"):
        st.session_state.prompt = "Create a Verilog module for a UART transmitter with 8 data bits, no parity."
    if st.button("⚡ SPI slave"):
        st.session_state.prompt = "Implement a SPI slave interface with 8-bit data width."

# ------------------------------
# Main content area
# ------------------------------
# Initialize session state for prompt if not exists
if "prompt" not in st.session_state:
    st.session_state.prompt = ""

# Input area
st.markdown("### 📝 Describe your hardware module")
prompt = st.text_area(
    "Enter your requirements in natural language",
    value=st.session_state.prompt,
    height=100,
    placeholder="e.g., Write a Verilog module for a 4-bit counter with synchronous reset and enable.",
    label_visibility="collapsed"
)

# Generate button and dummy generation
col_btn, col_spacer = st.columns([1, 5])
with col_btn:
    generate_clicked = st.button("🚀 Generate RTL", type="primary")

# Placeholder for output
output_placeholder = st.empty()

# Mock generation function (replace with actual model inference)
def generate_verilog(prompt_text):
    # Simulate processing delay
    time.sleep(1.5)
    # Return a simple Verilog module based on keywords in the prompt
    if "counter" in prompt_text.lower():
        return """// 4-bit Synchronous Counter with Active-Low Reset
module counter (
    input clk,
    input rst_n,
    input en,
    output reg [3:0] count
);
    always @(posedge clk or negedge rst_n) begin
        if (!rst_n)
            count <= 4'b0;
        else if (en)
            count <= count + 1;
    end
endmodule"""
    elif "alu" in prompt_text.lower():
        return """// 8-bit ALU with Add, Subtract, AND, OR
module alu (
    input [7:0] a, b,
    input [1:0] op,
    output reg [7:0] result
);
    always @(*) begin
        case(op)
            2'b00: result = a + b;
            2'b01: result = a - b;
            2'b10: result = a & b;
            2'b11: result = a | b;
            default: result = 8'b0;
        endcase
    end
endmodule"""
    elif "uart" in prompt_text.lower():
        return """// Simple UART Transmitter (8N1)
module uart_tx (
    input clk,
    input rst_n,
    input [7:0] data,
    input start,
    output reg tx,
    output reg busy
);
    // Implementation omitted for brevity
    // ...
endmodule"""
    elif "spi" in prompt_text.lower():
        return """// SPI Slave Interface
module spi_slave (
    input clk,
    input sclk,
    input mosi,
    input cs,
    output miso,
    input [7:0] data_in,
    output reg [7:0] data_out
);
    // Implementation omitted for brevity
    // ...
endmodule"""
    else:
        return """// Generated Verilog module
module custom_module (
    input clk,
    input rst_n,
    input [3:0] input_data,
    output reg [3:0] output_data
);
    always @(posedge clk or negedge rst_n) begin
        if (!rst_n)
            output_data <= 4'b0;
        else
            output_data <= input_data;
    end
endmodule"""

# Handle generation
if generate_clicked and prompt.strip() != "":
    with st.spinner("🧠 Semicon.ai is thinking..."):
        verilog_code = generate_verilog(prompt)
    output_placeholder.code(verilog_code, language="verilog", line_numbers=True)
elif generate_clicked:
    st.warning("Please enter a description first.")

# If there's already code shown (e.g., after generation) keep it; else show placeholder
else:
    output_placeholder.markdown("*Your generated RTL will appear here.*")

# ------------------------------
# Footer with name and roll number (as in PDF)
# ------------------------------
st.markdown("""
<div class="footer">
    Name: Sparsh Pandit | Roll No.: 23035010538<br>
    Department: Mehta Family School of Data Science & Artificial Intelligence<br>
    Institute: Indian Institute of Technology Guwahati<br>
    Email: sparsh.pandit@opiiitg.ac.in
</div>
""", unsafe_allow_html=True)

In [ ]:
!pip install streamlit

app_code = """
# Install Streamlit
# This is already installed by a previous cell, but included for completeness if run standalone.
# !pip install streamlit

import streamlit as st
import time

# ------------------------------
# Page configuration
# ------------------------------
st.set_page_config(
    page_title="Semicon.ai MVP",
    page_icon="🧠",
    layout="wide",
    initial_sidebar_state="expanded"
)

# ------------------------------
# Custom CSS for a polished look
# ------------------------------
st.markdown('''
<style>
    .main-header {
        font-size: 3rem;
        font-weight: 700;
        color: #1E3A8A;
        text-align: center;
        margin-bottom: 0.2rem;
    }
    .sub-header {
        font-size: 1.2rem;
        color: #4B5563;
        text-align: center;
        margin-top: 0rem;
        margin-bottom: 2rem;
    }
    .metric-card {
        background-color: #F3F4F6;
        border-radius: 10px;
        padding: 1.5rem;
        text-align: center;
        box-shadow: 0 2px 4px rgba(0,0,0,0.1);
    }
    .metric-value {
        font-size: 2.5rem;
        font-weight: 700;
        color: #1E3A8A;
    }
    .metric-label {
        font-size: 1rem;
        color: #6B7280;
        margin-top: 0.3rem;
    }
    .footer {
        text-align: center;
        margin-top: 3rem;
        padding-top: 1rem;
        border-top: 1px solid #E5E7EB;
        color: #9CA3AF;
        font-size: 0.9rem;
    }
    .stButton>button {
        background-color: #1E3A8A;
        color: white;
        font-weight: 600;
        border-radius: 8px;
        padding: 0.5rem 2rem;
        border: none;
        transition: all 0.2s;
    }
    .stButton>button:hover {
        background-color: #2563EB;
        color: white;
        border: none;
    }
    .example-button {
        background-color: #EFF6FF;
        border: 1px solid #BFDBFE;
        border-radius: 20px;
        padding: 0.3rem 1rem;
        margin: 0.2rem;
        font-size: 0.8rem;
        cursor: pointer;
        color: #1E3A8A;
        display: inline-block;
    }
    .example-button:hover {
        background-color: #DBEAFE;
    }
</style>
'''
, unsafe_allow_html=True)

# ------------------------------
# Header
# ------------------------------
st.markdown('<div class="main-header">Semicon.ai</div>', unsafe_allow_html=True)
st.markdown('<div class="sub-header">Your AI Copilot for Hardware Design</div>', unsafe_allow_html=True)

# ------------------------------
# Sidebar with metrics (from PDF)
# ------------------------------
with st.sidebar:
    st.markdown("## 📊 Model Performance")
    col1, col2 = st.columns(2)
    with col1:
        st.markdown('<div class="metric-card">'
                    '<div class="metric-value">94.2%</div>'
                    '<div class="metric-label">Syntax Pass Rate</div>'
                    '</div>', unsafe_allow_html=True)
    with col2:
        st.markdown('<div class="metric-card">'
                    '<div class="metric-value">110x</div>'
                    '<div class="metric-label">Speedup vs. Human</div>'
                    '</div>', unsafe_allow_html=True)

    st.markdown("---")
    st.markdown("### 🧪 Example Prompts")
    if st.button("🔁 4-bit counter"):
        st.session_state.prompt = "Write a Verilog module for a 4-bit synchronous counter with active-low reset."
    if st.button("➕ Simple ALU"):
        st.session_state.prompt = "Design an 8-bit ALU that supports add, subtract, AND, OR."
    if st.button("📟 UART transmitter"):
        st.session_state.prompt = "Create a Verilog module for a UART transmitter with 8 data bits, no parity."
    if st.button("⚡ SPI slave"):
        st.session_state.prompt = "Implement a SPI slave interface with 8-bit data width."

# ------------------------------
# Main content area
# ------------------------------
# Initialize session state for prompt if not exists
if "prompt" not in st.session_state:
    st.session_state.prompt = ""

# Input area
st.markdown("### 📝 Describe your hardware module")
prompt = st.text_area(
    "Enter your requirements in natural language",
    value=st.session_state.prompt,
    height=100,
    placeholder="e.g., Write a Verilog module for a 4-bit counter with synchronous reset and enable.",
    label_visibility="collapsed"
)

# Generate button and dummy generation
col_btn, col_spacer = st.columns([1, 5])
with col_btn:
    generate_clicked = st.button("🚀 Generate RTL", type="primary")

# Placeholder for output
output_placeholder = st.empty()

# Mock generation function (replace with actual model inference)
def generate_verilog(prompt_text):
    # Simulate processing delay
    time.sleep(1.5)
    # Return a simple Verilog module based on keywords in the prompt
    if "counter" in prompt_text.lower():
        return '''// 4-bit Synchronous Counter with Active-Low Reset
module counter (
    input clk,
    input rst_n,
    input en,
    output reg [3:0] count
);
    always @(posedge clk or negedge rst_n) begin
        if (!rst_n)
            count <= 4\'b0;
        else if (en)
            count <= count + 1;
    end
endmodule'''
    elif "alu" in prompt_text.lower():
        return '''// 8-bit ALU with Add, Subtract, AND, OR
module alu (
    input [7:0] a, b,
    input [1:0] op,
    output reg [7:0] result
);
    always @(*) begin
        case(op)
            2'b00: result = a + b;
            2'b01: result = a - b;
            2'b10: result = a & b;
            2'b11: result = a | b;
            default: result = 8\'b0;
        endcase
    end
endmodule'''
    elif "uart" in prompt_text.lower():
        return '''// Simple UART Transmitter (8N1)
module uart_tx (
    input clk,
    input rst_n,
    input [7:0] data,
    input start,
    output reg tx,
    output reg busy
);
    // Implementation omitted for brevity
    // ...
endmodule'''
    elif "spi" in prompt_text.lower():
        return '''// SPI Slave Interface
module spi_slave (
    input clk,
    input sclk,
    input mosi,
    input cs,
    output miso,
    input [7:0] data_in,
    output reg [7:0] data_out
);
    // Implementation omitted for brevity
    // ...
endmodule'''
    else:
        return '''// Generated Verilog module
module custom_module (
    input clk,
    input rst_n,
    input [3:0] input_data,
    output reg [3:0] output_data
);
    always @(posedge clk or negedge rst_n) begin
        if (!rst_n)
            output_data <= 4\'b0;
        else
            output_data <= input_data;
    end
endmodule'''

# Handle generation
if generate_clicked and prompt.strip() != "":
    with st.spinner("🧠 Semicon.ai is thinking..."):
        verilog_code = generate_verilog(prompt)
    output_placeholder.code(verilog_code, language="verilog", line_numbers=True)
elif generate_clicked:
    st.warning("Please enter a description first.")

# If there's already code shown (e.g., after generation) keep it; else show placeholder
else:
    output_placeholder.markdown("*Your generated RTL will appear here.*")

# ------------------------------
# Footer with name and roll number (as in PDF)
# ------------------------------
st.markdown('''
<div class="footer">
    Name: Sparsh Pandit | Roll No.: 23035010538<br>
    Department: Mehta Family School of Data Science & Artificial Intelligence<br>
    Institute: Indian Institute of Technology Guwahati<br>
    Email: sparsh.pandit@opiiitg.ac.in
</div>
'''
, unsafe_allow_html=True)
"""

with open("semicon_app.py", "w") as f:
    f.write(app_code)

!python -m streamlit run semicon_app.py